# 向量矩阵与张量运算

学习目标：区分逐元素乘法与矩阵乘法，判断一维和批量输入的结果形状，使用转置、迹与范数；选学指定轴的收缩计算。

前置知识：向量、矩阵、点积、转置、数组形状与广播。

运行环境：Python 3.12、NumPy 2.5。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

首个代码单元导入 NumPy，后续单元沿用 np。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 用矩阵计算多组加权结果

两条记录各有 3 个数值，同时按两组权重计算加权和。data 的行表示记录、列表示数值；weights 的每一列是一组权重。形状 (2, 3) 与 (3, 2) 相乘，结果为 (2, 2)。

先看结果左上角从哪里来：取 data 第一行，与 weights 第一列逐项相乘再求和，其余位置按相同方式组合。

![矩阵乘法中第一行 1、2、3 与第一列 1、0、1 配对，求和为结果左上角的 4。](image/illustration/13-01-matrix-product.svg)

共同的长度 3 用于求和，结果保留“哪条记录”和“哪组权重”两个轴。下面逐项对照 scores；图中只展开一个元素，代码一次计算整个矩阵。

用 ndarray 表达向量和矩阵，用 @ 计算矩阵乘积。NumPy 已不推荐 matrix 类；它的 * 表示矩阵乘法，容易与 ndarray 的逐元素乘法混淆。

In [1]:
import numpy as np

data = np.array([[1, 2, 3], [4, 5, 6]])
weights = np.array([[1, 0], [0, 1], [1, 1]])
scores = data @ weights

print(scores)
print(scores.shape)
# [[4, 5], [10, 11]]：每行对应一条记录，每列对应一组权重。

[[ 4  5]
 [10 11]]
(2, 2)


## 2 逐元素乘法与矩阵乘法

对 ndarray，a * b 和 np.multiply(a, b) 按元素相乘，并遵守广播规则。a @ b 和 np.matmul(a, b) 计算矩阵乘积。

二维矩阵乘法把左侧的一行与右侧的一列对应相乘后相加。若 a 的形状是 (m, k)，b 的形状是 (k, n)，结果形状为 (m, n)；m 是左侧行数，k 是相乘求和的长度，n 是右侧列数。

In [2]:
data = np.array([[1, 2, 3], [4, 5, 6]])
weights = np.array([[1, 0], [0, 1], [1, 1]])

print(data * weights[:, 0])
# 第一组权重逐元素作用于每行，仍保留 3 个分量。
manual = np.array([
    [1 * 1 + 2 * 0 + 3 * 1, 1 * 0 + 2 * 1 + 3 * 1],
    [4 * 1 + 5 * 0 + 6 * 1, 4 * 0 + 5 * 1 + 6 * 1],
])
print(manual)
print(np.matmul(data, weights))
# 手算与矩阵乘法都得到 [[4, 5], [10, 11]]。

[[1 0 3]
 [4 0 6]]
[[ 4  5]
 [10 11]]
[[ 4  5]
 [10 11]]


## 3 一维向量的乘积

两个等长一维实数数组的点积，是对应分量乘积的和，结果是标量。@ 和 dot 都能完成这个计算。

一维数组没有单独的行轴或列轴，不能把形状 (3,) 当成 (1, 3) 或 (3, 1)。一维数组的 .T 不增加轴；要明确行、列方向，需用 newaxis 增加轴。

In [3]:
u = np.array([1, 2, 3])
v = np.array([4, 5, 6])

print(u * v)  # 预期：[4 10 18]。
print(1 * 4 + 2 * 5 + 3 * 6)  # 预期：32，逐项乘积之和。
print(u @ v, np.dot(u, v))
print((u @ v).shape)
# 逐元素结果为 [4, 10, 18]；点积为 32，标量形状为 ()。
print(u.shape, u.T.shape)
print(u[np.newaxis, :].shape, u[:, np.newaxis].shape)
# 一维转置仍为 (3,)；显式行、列形状分别为 (1, 3)、(3, 1)。

[ 4 10 18]
32
32 32
()
(3,) (3,)
(1, 3) (3, 1)


matmul 遇到左侧一维输入，会临时在前面补一个长度为 1 的轴；遇到右侧一维输入，会在末尾补一个长度为 1 的轴。完成乘法后，只移除为一维输入补出的轴。显式写出的长度为 1 的轴会保留。

In [4]:
a = np.array([[1, 2, 3], [4, 5, 6]])
w = np.array([1, 0, 1])
row_weights = np.array([1, 2])

print(a @ w, (a @ w).shape)
print(a @ w[:, np.newaxis], (a @ w[:, np.newaxis]).shape)
# (2, 3) @ (3,) 得到 (2,)；显式列向量得到 (2, 1)。
print(row_weights @ a, (row_weights @ a).shape)
# (2,) @ (2, 3) 得到 [9, 12, 15]，形状为 (3,)。
print((w[np.newaxis, :] @ w[:, np.newaxis]).shape)
# 两个显式二维输入的结果仍为二维，形状为 (1, 1)。

[ 4 10] (2,)
[[ 4]
 [10]] (2, 1)
[ 9 12 15] (3,)
(1, 1)


## 4 批量矩阵乘法

多于二维时，matmul 把末尾两个轴当作矩阵的行、列轴，前面的轴当作批量轴，并对批量轴广播。矩阵内相乘求和的长度仍须相等。

下面有 2 个批次，每批 3 条记录，每条记录 4 个数值，形状为 (2, 3, 4)。所有批次共用一个 (4, 2) 权重矩阵，结果为 (2, 3, 2)。

In [5]:
batches = np.arange(24).reshape(2, 3, 4)
shared_weights = np.array([[1, 0], [0, 1], [1, 0], [0, 1]])
result = batches @ shared_weights

print(result.shape)
print(result[0])
# 首批结果为 [[2, 4], [10, 12], [18, 20]]。
print(batches[0] @ shared_weights)
# 单独计算首批，得到同样的 3 行结果。

(2, 3, 2)
[[ 2  4]
 [10 12]
 [18 20]]
[[ 2  4]
 [10 12]
 [18 20]]


如果每个批次使用各自的权重，将权重组织为 (2, 4, 2)。此时两个批量轴按位置配对，结果仍为 (2, 3, 2)。

In [6]:
batches = np.arange(24).reshape(2, 3, 4)
first_weights = np.array([[1, 0], [0, 1], [1, 0], [0, 1]])
batch_weights = np.stack([first_weights, first_weights * 2])
paired = batches @ batch_weights

print(batch_weights.shape, paired.shape)  # 预期：(2, 4, 2) (2, 3, 2)，两批分别相乘。
print(paired[1])
print(batches[1] @ batch_weights[1])
# 第 2 批只使用第 2 组权重，结果为 [[52, 56], [68, 72], [84, 88]]。

(2, 4, 2) (2, 3, 2)
[[52 56]
 [68 72]
 [84 88]]
[[52 56]
 [68 72]
 [84 88]]


## 5 dot 的维数规则

dot 的含义随输入维数变化，不能仅凭二维示例把它当成 matmul 的通用替代品。

| 输入条件 | dot 的计算 |
| --- | --- |
| 两个一维数组 | 对应分量相乘后求和，不取复共轭 |
| 两个二维数组 | 矩阵乘法 |
| 任意一个输入为零维 | 逐元素乘法，通常直接用 * |
| 右侧为一维数组 | 左侧最后一轴与右侧唯一轴相乘求和 |
| 两侧均非零维，右侧至少二维 | 左侧最后一轴与右侧倒数第二轴相乘求和，保留双方其余轴 |

对于上一节的两个三维输入，dot 保留双方的批量轴，得到 (2, 3, 2, 2)。其四个轴依次表示左批次、记录、右批次、输出列；matmul 则按批次配对。

In [7]:
batches = np.arange(24).reshape(2, 3, 4)
base_weights = np.array([[1, 0], [0, 1], [1, 0], [0, 1]])
batch_weights = np.stack([base_weights, base_weights * 2])
all_pairs = np.dot(batches, batch_weights)

print((batches @ batch_weights).shape)  # 预期：(2, 3, 2)。
print(all_pairs.shape)  # 预期：(2, 3, 2, 2)，包含左右批次的全部组合。
print(all_pairs[0, :, 1, :])
print(batches[0] @ batch_weights[1])
# dot 还计算了左侧第 1 批与右侧第 2 批的组合。
print(np.dot(np.array([1, 2, 3]), np.array([4, 5, 6])))
print(np.dot(np.array(2), np.array([1, 2, 3])))
# 一维点积为 32；零维输入使后一个调用得到 [2, 4, 6]。

(2, 3, 2)
(2, 3, 2, 2)
[[ 4  8]
 [20 24]
 [36 40]]
[[ 4  8]
 [20 24]
 [36 40]]
32
[2 4 6]


## 6 形状不兼容

matmul 的矩阵内轴不能用长度为 1 的广播来补齐；只有批量轴遵守广播规则。标量缩放使用 *，matmul 不接受零维输入。

In [8]:
# 预期 ValueError：矩阵乘法的内维分别为 3、1，不匹配。
np.ones((2, 3)) @ np.ones((1, 2))

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 1 is different from 3)

In [9]:
# 预期 ValueError：矩阵内维均为 4，但批次轴长度 2 与 5 不能广播。
np.ones((2, 3, 4)) @ np.ones((5, 4, 2))

ValueError: operands could not be broadcast together with remapped shapes [original->remapped]: (2,3,4)->(2,newaxis,newaxis) (5,4,2)->(5,newaxis,newaxis)  and requested shape (3,2)

In [10]:
# 预期 ValueError：右侧是零维标量，@ 的两个操作数都至少需要一维。
np.array([1, 2, 3]) @ np.array(2)

ValueError: matmul: Input operand 1 does not have enough dimensions (has 0, gufunc core with signature (n?,k),(k,m?)->(n?,m?) requires 1)

## 7 转置与共轭转置

二维数组的 .T 交换行列；高维数组的 .T 反转全部轴的顺序。批量矩阵只需交换末尾两个轴时，用 swapaxes(a, -1, -2)，保留前面的批量轴。

In [11]:
a = np.array([[1, 2, 3], [4, 5, 6]])
batches = np.arange(24).reshape(2, 3, 4)

print(a.T)  # 预期：三行分别为 [1, 4]、[2, 5]、[3, 6]。
print(batches.T.shape)  # 预期：(4, 3, 2)，三个轴的顺序反转。
matrix_transposes = np.swapaxes(batches, -1, -2)
print(matrix_transposes.shape)
print(matrix_transposes[0])
# .T 得到 (4, 3, 2)；只转置每个矩阵得到 (2, 4, 3)。

[[1 4]
 [2 5]
 [3 6]]
(4, 3, 2)
(2, 4, 3)
[[ 0  4  8]
 [ 1  5  9]
 [ 2  6 10]
 [ 3  7 11]]


复共轭把复数的虚部变号。二维矩阵的共轭转置同时做复共轭和转置，可写成 np.conjugate(a).T。实数数组的共轭转置与普通转置数值相同。

@ 和 dot 不会自动取复共轭。若计算约定要求先对左向量取共轭，必须明确写出；选学部分会介绍 vdot。

In [12]:
z = np.array([[1 + 2j, 3 - 1j, 2], [4j, 5, 6 + 1j]])
u = np.array([1 + 1j, 2])

print(z.T)
print(np.conjugate(z).T)
# 两种结果均为 (3, 2)，但虚部符号不同。
print(u @ u)
print(np.conjugate(u) @ u)
# 前者为 4 + 2j；后者为 (1 - 1j) * (1 + 1j) + 2 * 2，即 6。

[[1.+2.j 0.+4.j]
 [3.-1.j 5.+0.j]
 [2.+0.j 6.+1.j]]
[[1.-2.j 0.-4.j]
 [3.+1.j 5.-0.j]
 [2.-0.j 6.-1.j]]
(4+2j)
(6+0j)


## 8 迹与范数

### 8.1 对角线求和

方阵的迹（trace）是主对角线元素之和。np.trace 也能对矩形数组求对角线之和，默认使用轴 0 和轴 1。批量矩阵应明确指定末尾两个轴；被求和的两个轴从结果形状中移除。

In [13]:
a = np.array([[2, 1, 0], [4, 3, 1], [0, 2, 5]])
print(np.trace(a))
# 2 + 3 + 5 = 10。

batches = np.arange(24).reshape(2, 3, 4)
print(np.trace(batches, axis1=-2, axis2=-1))
print(np.trace(batches))
# 按每批矩阵求对角线和得到 [15, 51]，形状为 (2,)。
# 默认沿轴 0、1 求和得到 [16, 18, 20, 22]，保留的是原来的轴 2。

10
[15 51]
[16 18 20 22]


### 8.2 指定范数与轴

向量的 2-范数是各分量绝对值平方和的平方根；矩阵的 Frobenius 范数也对全部元素这样计算。np.linalg.norm 的 ord 指定范数：一维输入默认计算 2-范数，二维输入默认计算 Frobenius 范数。

axis 为整数时，沿该轴计算向量范数；axis 为两个轴组成的元组时，按这两个轴计算矩阵范数。keepdims=True 保留长度为 1 的被计算轴，方便广播。

In [14]:
v = np.array([3.0, 4.0])
a = np.array([[3.0, 0.0, 4.0], [0.0, 12.0, 0.0]])

print(np.linalg.norm(v))
print(np.linalg.norm(a, ord="fro"))
# 向量范数为 sqrt(9 + 16) = 5；矩阵范数为 sqrt(9 + 16 + 144) = 13。
row_norms = np.linalg.norm(a, axis=1, keepdims=True)
print(row_norms, row_norms.shape)
# 每行作为一个向量，结果为 [[5], [12]]，形状为 (2, 1)。
print(a / row_norms)
# 这里两行范数均非零，可以逐行除以范数。

5.0
13.0
[[ 5.]
 [12.]] (2, 1)
[[0.6 0.  0.8]
 [0.  1.  0. ]]


相同的 ord 在向量和矩阵上也可能表示不同计算。例如 ord=1 对向量计算绝对值之和，对矩阵计算各列绝对值之和的最大值。不要只记参数数字，还要确认输入维数和 axis。

In [15]:
a = np.array([[1, -2, 3], [-4, 5, -6]])
print(np.linalg.norm(a, ord=1))
print(np.linalg.norm(a, ord=1, axis=1))
# 矩阵 1-范数为 max(5, 7, 9) = 9；逐行向量 1-范数为 [6, 15]。

batches = np.stack([a, 2 * a])
print(np.linalg.norm(batches, ord="fro", axis=(-2, -1)).shape)
# 指定矩阵轴后，每批得到一个范数，形状为 (2,)。

9.0
[ 6. 15.]
(2,)


## 9 选学：其他内积与外积
vdot 对第一个参数取复共轭；多维输入会先展平，再得到一个点积。它不会逐批返回结果。

In [16]:
u = np.array([1 + 1j, 2])
print(np.vdot(u, u))
print(np.conjugate(u) @ u)
# 都得到 6 + 0j。

a = np.array([[1, 2, 3], [4, 5, 6]])
print(np.vdot(a, a))
print(np.sum(a * a))
# 两行一起参与计算，得到单个值 91。

(6+0j)
(6+0j)
91
91


inner 对两个输入的最后一轴相乘求和，不取复共轭，并依次保留双方其余轴。outer 计算两个向量每一对分量的乘积，没有求和；若输入不是一维，outer 会先展平。

In [17]:
a = np.array([[1, 2, 3], [4, 5, 6]])
b = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1], [1, 1, 1]])
inner_products = np.inner(a, b)
print(inner_products, inner_products.shape)
# (2, 3) 与 (4, 3) 的最后一轴收缩，得到 (2, 4)。

u = np.array([1, 2])
v = np.array([10, 20, 30])
print(np.outer(u, v))
# 外积为 [[10, 20, 30], [20, 40, 60]]，形状为 (2, 3)。

[[ 1  2  3  6]
 [ 4  5  6 15]] (2, 4)
[[10 20 30]
 [20 40 60]]


## 10 选学：指定轴收缩
把指定轴上的对应元素相乘后求和，称为轴收缩（axis contraction）。tensordot 用 axes 指定两侧配对求和的轴，每对轴长度须相等。结果先保留左侧未收缩的轴，再保留右侧未收缩的轴。

下面 readings 的轴依次是批次、通道、时刻，形状为 (2, 3, 4)。用长度为 3 的通道权重沿中间轴求加权和，保留批次和时刻。

In [18]:
readings = np.arange(24).reshape(2, 3, 4)
channel_weights = np.array([1, 0, 2])
weighted = np.tensordot(readings, channel_weights, axes=([1], [0]))

print(weighted, weighted.shape)
print(readings[:, 0, :] + 2 * readings[:, 2, :])
# 收缩长度为 3 的通道轴，得到 (2, 4)，与直接加权的结果相同。

[[16 19 22 25]
 [52 55 58 61]] (2, 4)
[[16 19 22 25]
 [52 55 58 61]]


einsum 用字母标记轴。显式写法在 -> 左侧列出输入的轴标签，右侧指定保留的标签及顺序；未保留的标签参与求和。

例如 ik,kj->ij 中，i、k 分别是左矩阵的行、列，k、j 分别是右矩阵的行、列；k 配对求和，输出按 i、j 排列。这就是二维矩阵乘法。指定轴更复杂时，再考虑这种写法。

In [19]:
a = np.array([[1, 2, 3], [4, 5, 6]])
b = np.array([[1, 0], [0, 1], [1, 1]])
product = np.einsum("ik,kj->ij", a, b)
print(product, product.shape)
print(a @ b)
# k 的长度为 3；输出 i、j 的长度均为 2。

readings = np.arange(24).reshape(2, 3, 4)
channel_weights = np.array([1, 0, 2])
print(np.einsum("bct,c->bt", readings, channel_weights))
# b、c、t 分别表示批次、通道、时刻，收缩 c，保留 b、t。

[[ 4  5]
 [10 11]] (2, 2)
[[ 4  5]
 [10 11]]
[[16 19 22 25]
 [52 55 58 61]]


## 本章小结

（1）ndarray 的 * 按元素计算，@ 表达矩阵乘法；先确定求和轴，再判断结果形状。

（2）一维输入会影响 matmul 的输出维数；批量输入的末尾两轴是矩阵轴，其余轴广播。

（3）dot 在高维输入下保留双方的非求和轴；复数计算是否取共轭必须明确。

（4）批量转置、trace 和 norm 都需要确认轴；选学的 tensordot、einsum 可直接指定收缩轴。

## 练习

（1）核对矩阵乘积。计算两条记录在两组权重下的结果。先手算结果的第一行，再用 @ 计算完整结果；另算第一组权重的逐元素乘积，说明它为什么还不是加权和。

In [20]:
records = np.array([[2, 1, 3], [0, 4, 2]])
weights = np.array([[1, 2], [2, 0], [0, 1]])
# 在此计算并打印结果、shape，以及第一组权重的逐元素乘积。
# 检查：输出的行对应记录，列对应权重组；写出第一行的手算过程。

（2）预测一维与二维结果。先预测下面三个结果的形状和值，再运行核对。解释为什么前两个结果的维数不同，以及为什么 v.T 没有成为列向量。

In [21]:
v = np.array([1, 2, 3])
matrix = np.array([[1, 0, 1], [0, 1, 1]])
# 在此写下预测，再运行下面的语句。
print(matrix @ v)
print(matrix @ v[:, np.newaxis])
print(v.T @ v)

[4 5]
[[4]
 [5]]
14


（3）权重条件改变后选择方法。两个批次原先共用 shared_weights，现在改为各用一组 batch_weights。分别完成计算，保持批次一一对应；在 @ 和 dot 中选择方法并说明理由。再说明新条件下误用另一种方法会保留哪些轴。

In [22]:
records = np.arange(24).reshape(2, 3, 4)
shared_weights = np.array([[1, 0], [0, 1], [1, 0], [0, 1]])
batch_weights = np.stack([shared_weights, 3 * shared_weights])
# 在此计算两种权重条件下的结果，打印 shape 和第二批结果。
# 检查：最终结果均应按“批次、记录、输出列”组织；写出选择理由。

（4）复数与范数。给定复向量 z，分别计算不取共轭的点积，以及先对左侧取共轭的点积。计算 z 的 2-范数，并用各分量绝对值平方和手算核对。说明这里应使用哪一种点积与手算值对照。

In [23]:
z = np.array([1 + 2j, 2 - 1j])
# 在此计算两种点积与向量范数。
# 检查：先算每个复数分量的绝对值平方，再相加；最后开平方得到范数。
# 可用 np.conjugate 明确写出共轭，不要求使用选学 API。

### 重点练习提示

对应第（3）题。先独立完成，再按需要查看提示。

（1）把最后两轴看作矩阵，先列出收缩的特征轴，再处理批次轴。

（2）比较 matmul 对批次的广播与 dot 对剩余轴的保留；用第二批首行手算确认批次没有交叉。

### 重点练习参考解析

对应第（3）题。

两种条件都可用 @：records @ shared_weights 将 (4, 2) 的权重用于每个批次；records @ batch_weights 让相同批次分别相乘。两者输出均为 (2, 3, 2)。共享权重时第二批为 [[26, 28], [34, 36], [42, 44]]；分批权重时这一批再乘 3，为 [[78, 84], [102, 108], [126, 132]]。

新条件下 dot(records, batch_weights) 的形状是 (2, 3, 2, 2)，依次保留数据批次、记录、权重批次和输出列。两个批次轴分别存在，包含交叉批次组合，不能直接当作一一对应的结果。

## 参考与引用来源

本章新增示意图由 CMYK Labs 原创，依据下表对应概念与本章教学输入绘制；示意图不作为实际运行截图或数学证明。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档（2.5） | [multiply](https://numpy.org/doc/2.5/reference/generated/numpy.multiply.html) 的 Parameters、Notes 与 Examples：逐元素乘法和广播；[matmul](https://numpy.org/doc/2.5/reference/generated/numpy.matmul.html) 的 Raises、Notes、Examples：矩阵乘法、一维提升、批量轴与复数；[dot](https://numpy.org/doc/2.5/reference/generated/numpy.dot.html) 的输入维数分类与 Examples；[matrix](https://numpy.org/doc/2.5/reference/generated/numpy.matrix.html) 的 Note：使用普通数组的建议；[ndarray.T](https://numpy.org/doc/2.5/reference/generated/numpy.ndarray.T.html)、[transpose](https://numpy.org/doc/2.5/reference/generated/numpy.transpose.html) 的维数说明与 Parameters、[swapaxes](https://numpy.org/doc/2.5/reference/generated/numpy.swapaxes.html) 的 Parameters：转置与轴交换；[conjugate](https://numpy.org/doc/2.5/reference/generated/numpy.conjugate.html) 的定义：复共轭；[trace](https://numpy.org/doc/2.5/reference/generated/numpy.trace.html) 的定义、Parameters：对角线和及轴；[linalg.norm](https://numpy.org/doc/2.5/reference/generated/numpy.linalg.norm.html) 的 Parameters、Notes 范数表与 Frobenius 公式：范数、axis、keepdims；[vdot](https://numpy.org/doc/2.5/reference/generated/numpy.vdot.html) 的定义、Examples：共轭与展平；[inner](https://numpy.org/doc/2.5/reference/generated/numpy.inner.html) 的 Returns、Notes：末轴收缩与输出形状；[outer](https://numpy.org/doc/2.5/reference/generated/numpy.outer.html) 的定义、Parameters：外积与展平；[tensordot](https://numpy.org/doc/2.5/reference/generated/numpy.tensordot.html) 的 Parameters、Notes：轴配对与结果顺序；[einsum](https://numpy.org/doc/2.5/reference/generated/numpy.einsum.html) 的 Parameters、Notes：显式标签与求和。 |